# Data Quality Checking for Stock Data
## Module 1: Foundations - ML for Engineers

---

**Learning Objectives:**
- Check for missing values
- Identify duplicate records
- Detect outliers
- Validate data types
- Verify data consistency
- Generate quality reports

**Why Data Quality Matters:**
- Garbage in, garbage out
- ML models learn from data quality
- Bad data = bad predictions

---

## 1. Setup and Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Load stock data
df = pd.read_csv('../datasets/AAPL_5y_SAMPLE.csv')
df['Date'] = pd.to_datetime(df['Date'])

print("✓ Data loaded")
print(f"Shape: {df.shape}")
df.head()

## 2. Missing Values Analysis

In [ ]:
# Check for missing values
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df)) * 100

missing_df = pd.DataFrame({
    'Column': missing.index,
    'Missing_Count': missing.values,
    'Missing_Percentage': missing_pct.values
})

print("="*60)
print("MISSING VALUES REPORT")
print("="*60)
print(missing_df.to_string(index=False))
print("\nTotal missing values:", missing.sum())

if missing.sum() == 0:
    print("✓ NO MISSING VALUES - Data quality is GOOD")
else:
    print("⚠ MISSING VALUES DETECTED - Needs attention")

In [ ]:
# Visualize missing values
if missing.sum() > 0:
    plt.figure(figsize=(10, 6))
    sns.heatmap(df.isnull(), cbar=False, cmap='viridis')
    plt.title('Missing Values Heatmap')
    plt.tight_layout()
    plt.show()

## 3. Duplicate Records Check

In [ ]:
# Check for duplicate dates
duplicate_dates = df['Date'].duplicated().sum()
print("="*60)
print("DUPLICATE DATES CHECK")
print("="*60)
print(f"Duplicate dates found: {duplicate_dates}")

if duplicate_dates > 0:
    print("\n⚠ DUPLICATES FOUND:")
    print(df[df['Date'].duplicated(keep=False)].sort_values('Date'))
else:
    print("✓ NO DUPLICATE DATES - Data quality is GOOD")

# Check for completely duplicate rows
duplicate_rows = df.duplicated().sum()
print(f"\nDuplicate rows (all columns): {duplicate_rows}")

if duplicate_rows > 0:
    print("⚠ DUPLICATE ROWS FOUND")
else:
    print("✓ NO DUPLICATE ROWS")

## 4. Data Type Validation

In [ ]:
print("="*60)
print("DATA TYPES VALIDATION")
print("="*60)

expected_types = {
    'Date': 'datetime64[ns]',
    'Open': 'float',
    'High': 'float',
    'Low': 'float',
    'Close': 'float',
    'Adj Close': 'float',
    'Volume': 'int'
}

type_issues = []
for col, expected in expected_types.items():
    if col in df.columns:
        actual = str(df[col].dtype)
        status = "✓" if expected in actual else "✗"
        print(f"{col:15} Expected: {expected:20} Actual: {actual:20} {status}")
        if expected not in actual:
            type_issues.append(col)

if type_issues:
    print(f"\n⚠ Type issues in: {', '.join(type_issues)}")
else:
    print("\n✓ ALL DATA TYPES CORRECT")

## 5. Outlier Detection

In [ ]:
# Statistical outlier detection using IQR method
def detect_outliers_iqr(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = data[(data[column] < lower_bound) | (data[column] > upper_bound)]
    return outliers, lower_bound, upper_bound

print("="*60)
print("OUTLIER DETECTION (IQR Method)")
print("="*60)

for col in ['Adj Close', 'Volume']:
    outliers, lower, upper = detect_outliers_iqr(df, col)
    print(f"\n{col}:")
    print(f"  Valid range: {lower:.2f} to {upper:.2f}")
    print(f"  Outliers found: {len(outliers)} ({len(outliers)/len(df)*100:.1f}%)")

    if len(outliers) > 0:
        print(f"  Sample outliers:")
        print(outliers[['Date', col]].head())

In [ ]:
# Box plots for outlier visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].boxplot(df['Adj Close'])
axes[0].set_title('Price Outliers')
axes[0].set_ylabel('Price (USD)')

axes[1].boxplot(df['Volume'])
axes[1].set_title('Volume Outliers')
axes[1].set_ylabel('Volume (shares)')

plt.tight_layout()
plt.show()

## 6. Data Consistency Checks

In [ ]:
print("="*60)
print("DATA CONSISTENCY CHECKS")
print("="*60)

# Check 1: High >= Low
invalid_high_low = df[df['High'] < df['Low']]
print(f"\n1. High < Low (invalid): {len(invalid_high_low)}")
if len(invalid_high_low) > 0:
    print("   ✗ CONSISTENCY ERROR")
else:
    print("   ✓ PASS")

# Check 2: High >= Open, Close
invalid_high = df[(df['High'] < df['Open']) | (df['High'] < df['Close'])]
print(f"\n2. High < Open or Close: {len(invalid_high)}")
if len(invalid_high) > 0:
    print("   ✗ CONSISTENCY ERROR")
else:
    print("   ✓ PASS")

# Check 3: Low <= Open, Close
invalid_low = df[(df['Low'] > df['Open']) | (df['Low'] > df['Close'])]
print(f"\n3. Low > Open or Close: {len(invalid_low)}")
if len(invalid_low) > 0:
    print("   ✗ CONSISTENCY ERROR")
else:
    print("   ✓ PASS")

# Check 4: Volume > 0
invalid_volume = df[df['Volume'] <= 0]
print(f"\n4. Volume <= 0: {len(invalid_volume)}")
if len(invalid_volume) > 0:
    print("   ✗ CONSISTENCY ERROR")
else:
    print("   ✓ PASS")

# Check 5: Prices > 0
price_cols = ['Open', 'High', 'Low', 'Close', 'Adj Close']
invalid_prices = df[(df[price_cols] <= 0).any(axis=1)]
print(f"\n5. Any price <= 0: {len(invalid_prices)}")
if len(invalid_prices) > 0:
    print("   ✗ CONSISTENCY ERROR")
else:
    print("   ✓ PASS")

## 7. Date Continuity Check

In [ ]:
print("="*60)
print("DATE CONTINUITY CHECK")
print("="*60)

# Sort by date
df_sorted = df.sort_values('Date').reset_index(drop=True)

# Calculate gaps between trading days
df_sorted['Days_Gap'] = df_sorted['Date'].diff().dt.days

# Expected gaps (weekends = 3 days, holidays can be more)
normal_gaps = df_sorted['Days_Gap'].value_counts().head(5)
print("\nMost common gaps between trading days:")
print(normal_gaps)

# Find unusual gaps (> 7 days)
large_gaps = df_sorted[df_sorted['Days_Gap'] > 7][['Date', 'Days_Gap']]
print(f"\nLarge gaps (> 7 days): {len(large_gaps)}")
if len(large_gaps) > 0:
    print("Dates with large gaps:")
    print(large_gaps)
else:
    print("✓ No unusual gaps detected")

## 8. Statistical Validation

In [ ]:
print("="*60)
print("STATISTICAL VALIDATION")
print("="*60)

# Calculate daily returns
df_sorted['Daily_Return'] = df_sorted['Adj Close'].pct_change() * 100

# Check for extreme daily movements (> 10%)
extreme_moves = df_sorted[abs(df_sorted['Daily_Return']) > 10]
print(f"\nExtreme daily movements (>10%): {len(extreme_moves)}")
if len(extreme_moves) > 0:
    print("\nDates with extreme movements:")
    print(extreme_moves[['Date', 'Adj Close', 'Daily_Return']].head(10))
    print("\n⚠ These may be valid (major news) or data errors")

# Daily return statistics
print("\nDaily Return Statistics:")
print(f"  Mean: {df_sorted['Daily_Return'].mean():.3f}%")
print(f"  Std Dev: {df_sorted['Daily_Return'].std():.3f}%")
print(f"  Max gain: {df_sorted['Daily_Return'].max():.3f}%")
print(f"  Max loss: {df_sorted['Daily_Return'].min():.3f}%")

## 9. Comprehensive Quality Report

In [ ]:
print("\n" + "="*70)
print("COMPREHENSIVE DATA QUALITY REPORT")
print("="*70)

total_checks = 10
passed_checks = 0
failed_checks = []

# 1. Missing values
if df.isnull().sum().sum() == 0:
    passed_checks += 1
    print("\n✓ 1. Missing Values: PASS (0 missing)")
else:
    failed_checks.append("Missing Values")
    print(f"\n✗ 1. Missing Values: FAIL ({df.isnull().sum().sum()} missing)")

# 2. Duplicate dates
if df['Date'].duplicated().sum() == 0:
    passed_checks += 1
    print("✓ 2. Duplicate Dates: PASS")
else:
    failed_checks.append("Duplicate Dates")
    print(f"✗ 2. Duplicate Dates: FAIL ({df['Date'].duplicated().sum()} duplicates)")

# 3-7. Data consistency
consistency_checks = [
    ("High >= Low", len(df[df['High'] < df['Low']]) == 0),
    ("Valid High", len(df[(df['High'] < df['Open']) | (df['High'] < df['Close'])]) == 0),
    ("Valid Low", len(df[(df['Low'] > df['Open']) | (df['Low'] > df['Close'])]) == 0),
    ("Positive Volume", len(df[df['Volume'] <= 0]) == 0),
    ("Positive Prices", len(df[(df[price_cols] <= 0).any(axis=1)]) == 0)
]

for i, (check_name, passed) in enumerate(consistency_checks, 3):
    if passed:
        passed_checks += 1
        print(f"✓ {i}. {check_name}: PASS")
    else:
        failed_checks.append(check_name)
        print(f"✗ {i}. {check_name}: FAIL")

# 8. Data types
if len(type_issues) == 0:
    passed_checks += 1
    print("✓ 8. Data Types: PASS")
else:
    failed_checks.append("Data Types")
    print(f"✗ 8. Data Types: FAIL ({len(type_issues)} issues)")

# 9. Date continuity
if len(large_gaps) == 0:
    passed_checks += 1
    print("✓ 9. Date Continuity: PASS")
else:
    failed_checks.append("Date Continuity")
    print(f"✗ 9. Date Continuity: FAIL ({len(large_gaps)} large gaps)")

# 10. Statistical sanity
if len(extreme_moves) < len(df) * 0.02:  # Less than 2% extreme moves
    passed_checks += 1
    print("✓ 10. Statistical Sanity: PASS")
else:
    failed_checks.append("Statistical Sanity")
    print(f"✗ 10. Statistical Sanity: WARNING ({len(extreme_moves)} extreme moves)")

# Summary
print("\n" + "="*70)
print("SUMMARY")
print("="*70)
print(f"Total checks: {total_checks}")
print(f"Passed: {passed_checks}")
print(f"Failed: {total_checks - passed_checks}")
print(f"\nQuality Score: {passed_checks/total_checks*100:.1f}%")

if passed_checks == total_checks:
    print("\n🎉 EXCELLENT DATA QUALITY - Ready for ML!")
elif passed_checks >= total_checks * 0.8:
    print("\n✓ GOOD DATA QUALITY - Minor issues to address")
    print(f"Issues: {', '.join(failed_checks)}")
else:
    print("\n⚠ DATA QUALITY NEEDS IMPROVEMENT")
    print(f"Issues: {', '.join(failed_checks)}")

print("="*70)

## 10. Export Quality Report

In [ ]:
# Create quality report dictionary
quality_report = {
    'Dataset': 'AAPL_5y',
    'Total_Rows': len(df),
    'Total_Columns': len(df.columns),
    'Date_Range': f"{df['Date'].min().date()} to {df['Date'].max().date()}",
    'Missing_Values': int(df.isnull().sum().sum()),
    'Duplicate_Dates': int(df['Date'].duplicated().sum()),
    'Duplicate_Rows': int(df.duplicated().sum()),
    'Checks_Passed': passed_checks,
    'Checks_Total': total_checks,
    'Quality_Score': f"{passed_checks/total_checks*100:.1f}%",
    'Status': 'PASS' if passed_checks == total_checks else 'NEEDS_REVIEW',
    'Issues': ', '.join(failed_checks) if failed_checks else 'None'
}

# Save to CSV
report_df = pd.DataFrame([quality_report])
report_df.to_csv('data_quality_report.csv', index=False)

print("✓ Quality report saved to: data_quality_report.csv")
print("\nReport Summary:")
for key, value in quality_report.items():
    print(f"{key:20}: {value}")

## Summary

### Data Quality Checks Performed:
1. Missing values detection
2. Duplicate records identification
3. Data type validation
4. Outlier detection (IQR method)
5. OHLC consistency checks
6. Price and volume validation
7. Date continuity analysis
8. Statistical sanity checks
9. Comprehensive quality scoring
10. Quality report generation

### Key Takeaways:
- Always check data quality before ML
- Document all quality issues
- Decide how to handle problems (fix, remove, or keep with caution)
- Re-validate after any data cleaning

---

**Course**: ML for Engineers - Module 1: Foundations  
**Created**: 2026-01-05  
**Version**: 1.0
